# Filtered C1 — Multires: physics λ sweep vs plain 2K / plain 60K (λ=0)

Protocol mirrors Model A in `data_c_amb_loss_diagnostics_NB.ipynb`, but pulses are the **spectrally filtered C1** family from `c1_pulse_independent_NB.ipynb`.

| Item | Value |
|------|-------|
| Pulses | filtered C1: $T=70$ fs, $N_{\mathrm{spikes}}=500$, $\sigma_{\mathrm{spike}}=1$ fs, $\sigma_{t,c}=5.73$ fs, $\Delta\omega$ on, FWHM filter $0.45$ eV |
| Canonicalize | `t0` (training) |
| Model | Multires (`TraceToPulseMultires`) |
| FROG | physical `FROGNet` ($N=64$) |
| Physics sweep | $\lambda \in \{1.5, 3.0, 4.5\}$ on **2K** train; **λ\*** = argmin val pulse L1 after best ambiguity |
| Plain 2K | $\lambda=0$, $n_{\mathrm{train}}=2048$ |
| Plain 60K | $\lambda=0$, $n_{\mathrm{train}}=60000$ |
| Val / test | 200 / 512 |
| Train SNR | U[0, 30] dB |
| Val SNR | U[0, 30] dB |
| Test SNR sweep | $[-10, 30]$ dB step 5 |
| Baselines | physics λ\*, plain 2K, **plain 60K**, PCGPA |

Artifacts: `checkpoints/benchmark/filtered_c1_multires_2k_diagnostics/`


In [ ]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch

SRC = Path.cwd()
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import data_c_amb_loss_diagnostics as diag
from data_generation import filtered_c1_pulse_config
from dataset_utils import build_filtered_c1_frog_dataloaders
from evaluate_cnn import load_cnn_sweep, plot_l1_snr_cnn_vs_pcgpa, plot_sim_snr_cnn_vs_pcgpa
from frog_reconstruction_model import extract_pulse_prediction
from pcgpa_reconstruct import mean_metrics_at_snr_pcgpa, reconstruct_pcgpa, _pcgpa_rng_for_pulse
from pulse_metrics import (
    best_l1_ambiguity,
    best_l1_ambiguity_field,
    best_similarity_error_ambiguity,
    pack_complex_field,
    prepare_frog_trace_for_plot,
    unpack_packed_field,
    unwrap_phases_for_overlay,
)
from trace_noise import add_trace_noise_awgn

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
OUT = Path("checkpoints/benchmark/filtered_c1_multires_2k_diagnostics")
OUT.mkdir(parents=True, exist_ok=True)

N_TRAIN, N_VAL, N_TEST = 2048, 200, 512
BATCH_SIZE = 64
SEED = 0
MAX_EPOCHS = 200
PATIENCE = 25
LR = 1e-3
TRAIN_SNR = (0.0, 30.0)
VAL_SNR = (0.0, 30.0)
SNR_SWEEP_DB = np.arange(-10.0, 31.0, 5.0)

LAM_SWEEP = [1.5, 3.0, 4.5]
LAM_PLAIN = 0.0
TAG_PLAIN = "filtered_c1_multires_lam0"
N_TRAIN_60K = 60000
TAG_PLAIN_60K = "filtered_c1_multires_60k_lam0"

RUN_TRAIN_PHYS = True
RUN_TRAIN_PLAIN = True
RUN_TRAIN_PLAIN_60K = True
FORCE_RETRAIN = False
FORCE_TEST_SWEEP = False
FORCE_PCGPA = False
PCGPA_MAXITER = 200
PCGPA_N_RESTARTS = 3
PCGPA_N_TEST = 32

print("device:", DEVICE)
print("OUT:", OUT.resolve())
print("pulse cfg:", filtered_c1_pulse_config(n=64))
print(f"plain 2K n_train={N_TRAIN}; plain 60K n_train={N_TRAIN_60K}")


## Physics λ sweep — train Multires with TRACE loss

In [ ]:
def tag_for_lam(lam: float) -> str:
    if float(lam) == 0.0:
        return "filtered_c1_multires_lam0"
    lam_s = f"{float(lam):g}".replace(".", "p")
    return f"filtered_c1_multires_lam{lam_s}"


def _train_or_load(tag, lam, *, n_train=None):
    n_train = int(N_TRAIN if n_train is None else n_train)
    hist_path = OUT / f"{tag}_history.npz"
    if hist_path.exists() and not FORCE_RETRAIN:
        print("skip train; using", hist_path)
        return None
    print(f"Training {tag} (lam={lam}, n_train={n_train}) on filtered C1...")
    result = diag.train_data_c_amb_diagnostics(
        pulse_loss_mode="raw",
        lam=float(lam),
        n_train=n_train,
        n_val=N_VAL,
        n_test=N_TEST,
        batch_size=BATCH_SIZE,
        seed=SEED,
        max_epochs=MAX_EPOCHS,
        patience=PATIENCE,
        lr=LR,
        train_snr_db_range=TRAIN_SNR,
        val_snr_db_range=VAL_SNR,
        device=DEVICE,
        verbose=True,
        ambiguity_backend="legacy",
        trace_loss_ref="clean",
        loader_builder="filtered_c1",
        canonicalize_mode="t0",
    )
    result_save = {k: v for k, v in result.items() if k != "bundle"}
    diag.save_run_artifacts(result_save, OUT, tag)
    print("saved", tag, "best_epoch=", result["best_epoch"], "best_score=", result["best_score"])
    return result


def _load_meta(tag):
    return json.loads((OUT / f"{tag}_meta.json").read_text(encoding="utf-8"))


phys_results = {}
for lam in LAM_SWEEP:
    tag = tag_for_lam(lam)
    if RUN_TRAIN_PHYS:
        phys_results[lam] = _train_or_load(tag, lam)

per_lam_meta = {}
for lam in LAM_SWEEP:
    tag = tag_for_lam(lam)
    meta = _load_meta(tag)
    per_lam_meta[lam] = meta
    print(f"λ={lam:g}  tag={tag}  best_score={meta['best_score']:.6f}")

LAM_PHYS = min(LAM_SWEEP, key=lambda lam: float(per_lam_meta[lam]["best_score"]))
TAG_PHYS = tag_for_lam(LAM_PHYS)
meta_phys = per_lam_meta[LAM_PHYS]
hist_phys = diag.load_history(OUT / f"{TAG_PHYS}_history.npz")

print(f"\nλ* = {LAM_PHYS:g}  TAG_PHYS = {TAG_PHYS}")
print(json.dumps(meta_phys, indent=2))


## Train Multires plain (λ=0)

In [ ]:
result_plain = _train_or_load(TAG_PLAIN, LAM_PLAIN) if RUN_TRAIN_PLAIN else None
hist_plain = diag.load_history(OUT / f"{TAG_PLAIN}_history.npz")
meta_plain = _load_meta(TAG_PLAIN)
print(json.dumps(meta_plain, indent=2))


## Plot helpers (Model A style; no timing figure / no trajectories)

In [ ]:
def plot_pulse_curves(hist, title_prefix):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(hist["train_pulse_l1_raw"], label="raw")
    axes[0].plot(hist["train_pulse_l1_amb"], label="best-amb")
    axes[0].set_title(f"{title_prefix}: train pulse L1")
    axes[0].set_xlabel("epoch")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    axes[1].plot(hist["val_pulse_l1_raw"], label="raw")
    axes[1].plot(hist["val_pulse_l1_amb"], label="best-amb")
    axes[1].set_title(f"{title_prefix}: val pulse L1 (SNR~U[0,30])")
    axes[1].set_xlabel("epoch")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_trace_curves(hist, title_prefix):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(hist["train_trace_l1"])
    axes[0].set_title(f"{title_prefix}: train TRACE L1")
    axes[0].set_xlabel("epoch")
    axes[0].grid(True, alpha=0.3)
    axes[1].plot(hist["val_trace_l1"])
    axes[1].set_title(f"{title_prefix}: val TRACE L1 (SNR~U[0,30])")
    axes[1].set_xlabel("epoch")
    axes[1].grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_grad_norms(hist, title_prefix, lam=0.0):
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.semilogy(hist["grad_norm_data"], label=r"$||\nabla L_{\mathrm{data}}||$")
    if float(lam) > 0.0:
        ax.semilogy(hist["grad_norm_reg"], label=r"$||\nabla L_{\mathrm{reg}}||$")
        if "grad_norm_total" in hist and len(hist["grad_norm_total"]):
            ax.semilogy(
                hist["grad_norm_total"],
                label=r"$||\nabla(L_{\mathrm{data}}+\lambda L_{\mathrm{reg}})||$",
            )
    else:
        ax.semilogy(
            hist["grad_norm_reg"],
            ls="--",
            alpha=0.75,
            label=r"$||\nabla L_{\mathrm{reg}}||$ (NOT in loss)",
        )
        if "grad_norm_total" in hist and len(hist["grad_norm_total"]):
            ax.semilogy(
                hist["grad_norm_total"],
                label=r"$||\nabla L_{\mathrm{total}}||$ (= data only)",
            )
    ax.set_title(f"{title_prefix}: gradient norms (end of epoch probe)")
    ax.set_xlabel("epoch")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()


def plot_grad_norms_linear_separate(hist, title_prefix, lam=0.0):
    if float(lam) > 0.0:
        keys = ["grad_norm_data", "grad_norm_reg", "grad_norm_total"]
        labels = [
            r"$||\nabla L_{\mathrm{data}}||$",
            r"$||\nabla L_{\mathrm{reg}}||$",
            r"$||\nabla L_{\mathrm{total}}||$",
        ]
    else:
        keys = ["grad_norm_data", "grad_norm_total"]
        labels = [
            r"$||\nabla L_{\mathrm{data}}||$",
            r"$||\nabla L_{\mathrm{total}}||$",
        ]
    fig, axes = plt.subplots(1, len(keys), figsize=(4.5 * len(keys), 4))
    if len(keys) == 1:
        axes = [axes]
    for ax, k, lab in zip(axes, keys, labels):
        ax.plot(hist[k])
        ax.set_title(lab)
        ax.set_xlabel("epoch")
        ax.grid(True, alpha=0.3)
    fig.suptitle(f"{title_prefix}: gradient norms (linear)")
    plt.tight_layout()
    plt.show()


def summarize_timings(hist, name):
    keys = [
        "timing_data_prep_sec",
        "timing_loss_data_fwd_sec",
        "timing_loss_reg_fwd_sec",
        "timing_total_backward_sec",
        "timing_optimizer_step_sec",
    ]
    print(f"=== {name}: mean±std over epochs (ms/batch) ===")
    for k in keys:
        if k not in hist:
            continue
        x = np.asarray(hist[k], dtype=float) * 1e3
        print(f"  {k.replace('timing_', ''):22s}  {x.mean():7.3f} ± {x.std():6.3f}")


assert hist_phys is not None and hist_plain is not None  # hist_plain_60k loaded later


## Physics (λ\*) plots — winner from sweep

In [ ]:
plot_pulse_curves(hist_phys, f"Physics λ*={LAM_PHYS:g}")
plot_trace_curves(hist_phys, f"Physics λ*={LAM_PHYS:g}")
plot_grad_norms(hist_phys, f"Physics λ*={LAM_PHYS:g}", lam=LAM_PHYS)
plot_grad_norms_linear_separate(hist_phys, f"Physics λ*={LAM_PHYS:g}", lam=LAM_PHYS)
summarize_timings(hist_phys, f"Physics λ*={LAM_PHYS:g}")

fig, ax = plt.subplots(figsize=(8, 4))
for lam in LAM_SWEEP:
    tag = tag_for_lam(lam)
    hist_l = diag.load_history(OUT / f"{tag}_history.npz")
    ax.plot(hist_l["val_pulse_l1_amb"], label=f"λ={lam:g}")
ax.set_xlabel("epoch")
ax.set_ylabel("val pulse L1 (best-amb)")
ax.set_title("Filtered C1 Multires — val best-amb L1 (all λ)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Plain (λ=0) plots

For λ=0 the training loss is **pulse L1 only**; the TRACE / physics term $L_{\mathrm{reg}}$ is **not** included in the objective (grad-norm panels omit $L_{\mathrm{reg}}$ from the training-loss view).


In [ ]:
plot_pulse_curves(hist_plain, "Plain λ=0")
plot_trace_curves(hist_plain, "Plain λ=0")
plot_grad_norms(hist_plain, "Plain λ=0", lam=0.0)
plot_grad_norms_linear_separate(hist_plain, "Plain λ=0", lam=0.0)
summarize_timings(hist_plain, "Plain λ=0")


## Train Multires **60K** plain (λ=0)

Same protocol as Multires 2K plain, but \(n_{\mathrm{train}}=60000\). Long data-generation + training run; skipped if artifacts exist (`FORCE_RETRAIN=False`).


In [ ]:
result_plain_60k = (
    _train_or_load(TAG_PLAIN_60K, LAM_PLAIN, n_train=N_TRAIN_60K)
    if RUN_TRAIN_PLAIN_60K
    else None
)
hist_plain_60k = diag.load_history(OUT / f"{TAG_PLAIN_60K}_history.npz")
meta_plain_60k = _load_meta(TAG_PLAIN_60K)
print(json.dumps(meta_plain_60k, indent=2))


## Plain 60K (λ=0) plots

Same figure suite as Multires 2K plain (pulse / TRACE monitors / grads with TRACE marked **NOT in loss**).


In [ ]:
plot_pulse_curves(hist_plain_60k, "Plain 60K λ=0")
plot_trace_curves(hist_plain_60k, "Plain 60K λ=0")
plot_grad_norms(hist_plain_60k, "Plain 60K λ=0", lam=0.0)
plot_grad_norms_linear_separate(hist_plain_60k, "Plain 60K λ=0", lam=0.0)
summarize_timings(hist_plain_60k, "Plain 60K λ=0")


## Overlay: val pulse L1 best-amb — physics λ\* / plain 2K / plain 60K


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(hist_phys["val_pulse_l1_amb"], label=f"physics λ*={LAM_PHYS:g} (2K)")
ax.plot(hist_plain["val_pulse_l1_amb"], label="plain 2K λ=0")
ax.plot(hist_plain_60k["val_pulse_l1_amb"], label="plain 60K λ=0")
ax.set_xlabel("epoch")
ax.set_ylabel("val pulse L1 (best-amb)")
ax.set_title("Filtered C1 Multires — validation best-amb L1")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Held-out test SNR sweep — Multires physics (λ\*) / plain 2K / plain 60K / PCGPA


In [ ]:
bundle = build_filtered_c1_frog_dataloaders(
    n_train=N_TRAIN,
    n_val=N_VAL,
    n_test=N_TEST,
    batch_size=BATCH_SIZE,
    seed=SEED,
    device=DEVICE,
    grid=filtered_c1_pulse_config(n=64),
    canonicalize_mode="t0",
)
test_loader = bundle.test_loader
t_axis = np.asarray(bundle.t_vec, dtype=float)
w_vec = np.asarray(bundle.w_vec, dtype=float)
dt = float(t_axis[1] - t_axis[0])
print("test batches:", len(test_loader), "| SNR grid:", SNR_SWEEP_DB.tolist())

sweep_phys_path = OUT / f"{TAG_PHYS}_test_snr_sweep.npz"
sweep_plain_path = OUT / f"{TAG_PLAIN}_test_snr_sweep.npz"
sweep_plain_60k_path = OUT / f"{TAG_PLAIN_60K}_test_snr_sweep.npz"

if FORCE_TEST_SWEEP or not sweep_phys_path.exists():
    print(f"Running test SNR sweep — physics λ*={LAM_PHYS:g}...")
    diag.run_and_save_test_snr_sweep(
        OUT / f"{TAG_PHYS}_model.pt",
        sweep_phys_path,
        test_loader=test_loader,
        snr_sweep_db=SNR_SWEEP_DB,
        device=DEVICE,
        experiment_name=f"Multires physics λ*={LAM_PHYS:g}",
    )
else:
    print("skip physics sweep; using", sweep_phys_path)

if FORCE_TEST_SWEEP or not sweep_plain_path.exists():
    print("Running test SNR sweep — plain 2K λ=0...")
    diag.run_and_save_test_snr_sweep(
        OUT / f"{TAG_PLAIN}_model.pt",
        sweep_plain_path,
        test_loader=test_loader,
        snr_sweep_db=SNR_SWEEP_DB,
        device=DEVICE,
        experiment_name="Multires plain 2K λ=0",
    )
else:
    print("skip plain 2K sweep; using", sweep_plain_path)

if FORCE_TEST_SWEEP or not sweep_plain_60k_path.exists():
    print("Running test SNR sweep — plain 60K λ=0...")
    diag.run_and_save_test_snr_sweep(
        OUT / f"{TAG_PLAIN_60K}_model.pt",
        sweep_plain_60k_path,
        test_loader=test_loader,
        snr_sweep_db=SNR_SWEEP_DB,
        device=DEVICE,
        experiment_name="Multires plain 60K λ=0",
    )
else:
    print("skip plain 60K sweep; using", sweep_plain_60k_path)

sweep_phys = load_cnn_sweep(sweep_phys_path)
sweep_plain = load_cnn_sweep(sweep_plain_path)
sweep_plain_60k = load_cnn_sweep(sweep_plain_60k_path)


In [ ]:
pcgpa_path = OUT / "filtered_c1_pcgpa_snr_sweep.npz"
I_test = test_loader.dataset.tensors[0].detach().cpu().numpy()
E_test = test_loader.dataset.tensors[1].detach().cpu().numpy()

if FORCE_PCGPA or not pcgpa_path.exists():
    print("Running PCGPA SNR sweep...")
    sim_m, sim_s, l1_m, l1_s = [], [], [], []
    for snr in SNR_SWEEP_DB:
        (sm, ss), (lm, ls) = mean_metrics_at_snr_pcgpa(
            I_test,
            E_test,
            float(snr),
            add_noise_fn=add_trace_noise_awgn,
            dt=dt,
            omega_axis=w_vec,
            maxiter=PCGPA_MAXITER,
            n_restarts=PCGPA_N_RESTARTS,
            n_subsample=PCGPA_N_TEST,
            seed=SEED,
            use_best_ambiguity=True,
            show_progress=True,
        )
        sim_m.append(sm)
        sim_s.append(ss)
        l1_m.append(lm)
        l1_s.append(ls)
        print(f"  SNR={snr:5.1f}  L1_amb={lm:.4f}±{ls:.4f}  SIM_amb={sm:.4f}±{ss:.4f}")
    np.savez_compressed(
        pcgpa_path,
        snr_db=SNR_SWEEP_DB,
        l1_amb_m=np.asarray(l1_m),
        l1_amb_s=np.asarray(l1_s),
        sim_amb_m=np.asarray(sim_m),
        sim_amb_s=np.asarray(sim_s),
    )
else:
    print("skip PCGPA sweep; using", pcgpa_path)

pcgpa = np.load(pcgpa_path)


In [ ]:
plot_l1_snr_cnn_vs_pcgpa(
    SNR_SWEEP_DB,
    [sweep_phys, sweep_plain, sweep_plain_60k],
    pcgpa_l1_m=pcgpa["l1_amb_m"],
    pcgpa_l1_s=pcgpa["l1_amb_s"],
    pcgpa_label=f"PCGPA (n={PCGPA_N_TEST})",
    include_vs_n=False,
)
plot_sim_snr_cnn_vs_pcgpa(
    SNR_SWEEP_DB,
    [sweep_phys, sweep_plain, sweep_plain_60k],
    pcgpa_sim_m=pcgpa["sim_amb_m"],
    pcgpa_sim_s=pcgpa["sim_amb_s"],
    pcgpa_label=f"PCGPA (n={PCGPA_N_TEST})",
    include_vs_n=False,
)


## Example reconstruction @ SNR = 10 dB

One random held-out test sample: **separate figure** per algorithm (physics λ\*, plain 2K, **plain 60K**, PCGPA) with clean TRACE, noisy TRACE, $|E(t)|$, and phase after `best_l1_ambiguity_field`. Metrics: `best_l1_ambiguity` and `best_similarity_error_ambiguity`.


In [ ]:
EXAMPLE_SNR_DB = 10.0

_test_ds = test_loader.dataset
EXAMPLE_INDEX = int(np.random.randint(0, len(_test_ds)))
I_clean, E_true_packed = _test_ds[EXAMPLE_INDEX]
I_clean = I_clean.to(DEVICE)
E_true_packed = E_true_packed.to(DEVICE)
I_noisy = add_trace_noise_awgn(I_clean.unsqueeze(0), EXAMPLE_SNR_DB).squeeze(0)
print(f"Drew test sample index {EXAMPLE_INDEX} / {len(_test_ds) - 1}")

model_phys = diag.load_trained_multires(OUT / f"{TAG_PHYS}_model.pt", device=DEVICE)
model_plain = diag.load_trained_multires(OUT / f"{TAG_PLAIN}_model.pt", device=DEVICE)
model_plain_60k = diag.load_trained_multires(OUT / f"{TAG_PLAIN_60K}_model.pt", device=DEVICE)

with torch.no_grad():
    x = I_noisy.unsqueeze(0).unsqueeze(0)
    E_phys = extract_pulse_prediction(model_phys(x)).squeeze(0).cpu().numpy()
    E_plain = extract_pulse_prediction(model_plain(x)).squeeze(0).cpu().numpy()
    E_plain_60k = extract_pulse_prediction(model_plain_60k(x)).squeeze(0).cpu().numpy()

e_true = unpack_packed_field(E_true_packed.detach().cpu().numpy())
e_phys = unpack_packed_field(E_phys)
e_plain = unpack_packed_field(E_plain)
e_plain_60k = unpack_packed_field(E_plain_60k)
e_pcgpa = reconstruct_pcgpa(
    I_noisy.detach().cpu().numpy(),
    dt=dt,
    maxiter=PCGPA_MAXITER,
    n_restarts=PCGPA_N_RESTARTS,
    rng=_pcgpa_rng_for_pulse(SEED, EXAMPLE_INDEX, EXAMPLE_SNR_DB),
    omega_axis=w_vec,
)

reconstructions = {
    f"Physics λ*={LAM_PHYS:g}": best_l1_ambiguity_field(e_phys, e_true),
    "Plain 2K λ=0": best_l1_ambiguity_field(e_plain, e_true),
    "Plain 60K λ=0": best_l1_ambiguity_field(e_plain_60k, e_true),
    "PCGPA": best_l1_ambiguity_field(e_pcgpa, e_true),
}
raw_fields = {
    f"Physics λ*={LAM_PHYS:g}": e_phys,
    "Plain 2K λ=0": e_plain,
    "Plain 60K λ=0": e_plain_60k,
    "PCGPA": e_pcgpa,
}

print(f"\nExample sample {EXAMPLE_INDEX} @ SNR={EXAMPLE_SNR_DB:.0f} dB")
print(f"{'Algorithm':<22} {'L1_amb':>10} {'SIM_amb':>10}")
print("-" * 44)
for name, e_raw in raw_fields.items():
    l1_amb = best_l1_ambiguity(e_raw, e_true)
    sim_amb = best_similarity_error_ambiguity(e_raw, e_true)
    print(f"{name:<22} {l1_amb:10.4f} {sim_amb:10.4f}")

trace_c, tau_axis, omega_plot = prepare_frog_trace_for_plot(
    I_clean.detach().cpu().numpy(), num_points=len(t_axis), dt=dt
)
trace_n, _, _ = prepare_frog_trace_for_plot(
    I_noisy.detach().cpu().numpy(), num_points=len(t_axis), dt=dt
)
energy_plot = omega_plot * 4.135667696 / (2.0 * np.pi)

for name, e_aligned in reconstructions.items():
    fig, axes = plt.subplots(2, 2, figsize=(12, 9))
    im0 = axes[0, 0].imshow(
        trace_c,
        origin="lower",
        aspect="auto",
        extent=[tau_axis[0], tau_axis[-1], energy_plot[0], energy_plot[-1]],
        cmap="magma",
    )
    axes[0, 0].set_title("Clean TRACE")
    axes[0, 0].set_xlabel("Delay τ [fs]")
    axes[0, 0].set_ylabel("Relative energy [eV]")
    fig.colorbar(im0, ax=axes[0, 0], fraction=0.046)

    im1 = axes[0, 1].imshow(
        trace_n,
        origin="lower",
        aspect="auto",
        extent=[tau_axis[0], tau_axis[-1], energy_plot[0], energy_plot[-1]],
        cmap="magma",
    )
    axes[0, 1].set_title(f"Noisy TRACE ({EXAMPLE_SNR_DB:.0f} dB)")
    axes[0, 1].set_xlabel("Delay τ [fs]")
    axes[0, 1].set_ylabel("Relative energy [eV]")
    fig.colorbar(im1, ax=axes[0, 1], fraction=0.046)

    axes[1, 0].plot(t_axis, np.abs(e_true), "k-", lw=2, label="true")
    axes[1, 0].plot(t_axis, np.abs(e_aligned), lw=1.5, label=name)
    axes[1, 0].set_title("|E(t)| after best-amb")
    axes[1, 0].set_xlabel("Time [fs]")
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)

    ph_true, ph_rec = unwrap_phases_for_overlay(e_aligned, e_true)
    axes[1, 1].plot(t_axis, ph_true, "k-", lw=2, label="true")
    axes[1, 1].plot(t_axis, ph_rec, lw=1.5, label=name)
    axes[1, 1].set_title("phase(E(t)) after best-amb")
    axes[1, 1].set_xlabel("Time [fs]")
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)

    plt.suptitle(
        f"Filtered C1 — {name} | sample {EXAMPLE_INDEX}, SNR={EXAMPLE_SNR_DB:.0f} dB",
        fontweight="bold",
    )
    plt.tight_layout()
    plt.show()
